# Módulo de Prompt Engineering — Entregable Final
## Análisis de reseñas de videojuegos con LLMs

**Objetivo:** Diseñar un pipeline en dos pasos con LLMs para:
1. Filtrar reseñas relevantes (LLM #1)
2. Extraer información estructurada de las reseñas útiles (LLM #2)

**Proveedor:** Groq API  
**Modelos utilizados:**
- LLM #1 (filtrado): `llama3-8b-8192` — modelo más ligero, tarea binaria simple
- LLM #2 (extracción): `llama3-70b-8192` — modelo más potente, tarea estructurada compleja

**Justificación del proveedor:** Groq ofrece inferencia ultra-rápida (LPU) con modelos open-source de alta calidad, tier gratuito generoso y latencia muy baja — ideal para pipelines con múltiples llamadas.

## 0. Instalación de dependencias

In [ ]:
!pip install groq pandas tqdm dotenv

## 1. Imports y configuración

In [ ]:
import os
import json
import time
import pandas as pd
from groq import Groq
from tqdm import tqdm

# ─── API Key ───────────────────────────────────────────────────────────────────
from dotenv import load_dotenv

load_dotenv()  # Lee el archivo .env y carga las variables

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)


# ─── Modelos ───────────────────────────────────────────────────────────────────
# LLM #1: tarea binaria simple → modelo ligero, más barato en tokens
MODEL_FILTER   = "llama-3.1-8b-instant"
# LLM #2: extracción estructurada compleja → modelo más potente
MODEL_EXTRACT  = "llama-3.3-70b-versatile"

# ─── Parámetros de batching ────────────────────────────────────────────────────
# Groq free tier: ~30 req/min para llama3-8b, ~14 req/min para llama3-70b
# Enviamos 5 reseñas por batch en el filtrado para:
#   - Mantener el prompt dentro del context window (8192 tokens)
#   - Reducir nº de llamadas (100 reseñas / 5 = 20 llamadas)
#   - Tener respuestas más controlables por LLM
BATCH_SIZE_FILTER  = 2
# En la extracción: 1 reseña por llamada para máxima precisión en el JSON
BATCH_SIZE_EXTRACT = 1

# Pausa entre llamadas para respetar rate limits
SLEEP_FILTER  = 2.5   # segundos entre batches de filtrado
SLEEP_EXTRACT = 4.5   # segundos entre llamadas de extracción

print("✅ Configuración lista")
print(f"   Modelo filtrado:    {MODEL_FILTER}")
print(f"   Modelo extracción:  {MODEL_EXTRACT}")
print(f"   Batch filtrado:     {BATCH_SIZE_FILTER} reseñas/llamada")
print(f"   Sleep filtrado:     {SLEEP_FILTER}s | Sleep extracción: {SLEEP_EXTRACT}s")

In [ ]:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Di 'API funcionando correctamente' y nada más."}],
    max_tokens=200
)

print(response.choices[0].message.content)

## 2. Carga del dataset y selección inicial (Paso 1)

Se carga el CSV y se seleccionan las **100 reseñas con mayor longitud de contenido**, tal como indica el enunciado. Este paso ya viene dado pero lo incluimos para reproducibilidad completa del pipeline.

In [ ]:
# Cargar el dataset
df_raw = pd.read_csv("input/videogames_reviews.csv")

print(f"Dataset original: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas")
print(f"Columnas: {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:

print("Forma:", df_raw.shape)
print("Columnas:", list(df_raw.columns))
print("\nNulos por columna:\n", df_raw.isnull().sum())
print("\nDistribución de 'Valoración':\n", df_raw["Valoración"].value_counts())
print("\nDistribución de 'Recomendado_binario':\n", df_raw["Recomendado_binario"].value_counts())

In [ ]:
# ─── Paso 1: Selección de las 100 reseñas más largas ───────────────────────────
df_raw["Contenido"] = df_raw["Contenido"].fillna("")
df_raw["content_length"] = df_raw["Contenido"].astype(str).apply(len)

df_top100 = (
    df_raw
    .sort_values("content_length", ascending=False)
    .head(100)
    .reset_index(drop=True)
)

# Añadimos un ID numérico limpio para el pipeline
df_top100["review_id"] = df_top100.index

print(f"\nSubconjunto seleccionado: {df_top100.shape[0]} reseñas")
print(f"Longitud media del contenido: {df_top100['content_length'].mean():.0f} caracteres")
print(f"Longitud mínima: {df_top100['content_length'].min()} | Máxima: {df_top100['content_length'].max()}")
df_top100[["review_id", "Contenido", "Valoración", "Recomendado_binario"]].head()

## 3. Paso 2 — Filtrado con LLM #1 (`llama-3.1-8b-instant`)

### Diseño del prompt de filtrado

**Decisiones de diseño:**
- Rol claro: *clasificador de calidad* de reseñas
- Tarea binaria: `RELEVANTE` / `IRRELEVANTE` → fácil de parsear
- Se incluyen ejemplos de lo que NO es útil (spam, texto vacío, sin semántica)
- Output en JSON array para parseo robusto
- Se mandan **5 reseñas por batch** para equilibrar eficiencia y control

**Por qué llama3-8b aquí:**  
La tarea es binaria y no requiere razonamiento complejo. El modelo ligero es más rápido, consume menos tokens y tiene rate limits más generosos.

In [ ]:
def build_filter_prompt(reviews_batch: list[dict]) -> str:
    """
    Construye el prompt para el LLM de filtrado.
    Recibe una lista de dicts con 'id' y 'texto'.
    Retorna el prompt completo como string.
    """
    reviews_text = ""
    for r in reviews_batch:
        reviews_text += f"\n---\nID: {r['id']}\nReseña: {r['texto']}\n"

    prompt = f"""Eres un clasificador experto en calidad de reseñas de videojuegos.

Tu tarea es analizar cada reseña y determinar si es RELEVANTE o IRRELEVANTE.

Una reseña es IRRELEVANTE si:
- Es spam, texto sin sentido o caracteres aleatorios
- Está vacía o casi vacía
- No expresa ninguna opinión sobre el videojuego
- Solo contiene emojis, símbolos o frases sin contenido semántico
- Es una broma sin información útil (ej: "jeje", "lol", "xd")

Una reseña es RELEVANTE si:
- Expresa una opinión sobre el juego (positiva, negativa o neutra)
- Menciona aspectos concretos: gameplay, historia, gráficos, dificultad, precio, etc.
- Tiene suficiente contenido para extraer información útil

RESEÑAS A CLASIFICAR:
{reviews_text}

Responde ÚNICAMENTE con un array JSON con este formato exacto, sin texto adicional:
[
  {{"id": <número>, "decision": "RELEVANTE" | "IRRELEVANTE", "razon": "<motivo breve>"}},
  ...
]"""
    return prompt


def call_llm_filter(reviews_batch: list[dict], retries: int = 3) -> list[dict]:
    """
    Llama al LLM de filtrado con manejo de errores y reintentos.
    Retorna lista de dicts con id, decision y razon.
    """
    prompt = build_filter_prompt(reviews_batch)

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_FILTER,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,   # Determinismo total: tarea clasificatoria
                max_tokens=512,    # Suficiente para el JSON de 5 clasificaciones
            )
            raw_text = response.choices[0].message.content.strip()

            # Parseo robusto: extraemos el JSON aunque haya texto alrededor
            start = raw_text.find("[")
            end   = raw_text.rfind("]") + 1
            json_str = raw_text[start:end]
            result = json.loads(json_str)
            return result

        except json.JSONDecodeError as e:
            print(f"  ⚠️  JSON inválido (intento {attempt+1}/{retries}): {e}")
            time.sleep(2)
        except Exception as e:
            print(f"  ❌ Error API (intento {attempt+1}/{retries}): {e}")
            time.sleep(5)

    # Si todos los reintentos fallan, marcamos las reseñas como RELEVANTE por defecto
    print(f"  ⚠️  Batch fallido. Marcando {len(reviews_batch)} reseñas como RELEVANTE por defecto.")
    return [{"id": r["id"], "decision": "RELEVANTE", "razon": "error_api"} for r in reviews_batch]


print("✅ Funciones de filtrado definidas")
print("\n--- Ejemplo de prompt generado ---")
ejemplo = [{"id": 0, "texto": "Este juego es increíble, los gráficos son una pasada y la historia engancha desde el minuto uno."}]
print(build_filter_prompt(ejemplo))

In [ ]:
# ─── Ejecución del pipeline de filtrado ────────────────────────────────────────

# Preparamos la lista de reseñas
reviews_list = [
    {"id": row["review_id"], "texto": str(row["Contenido"])}
    for _, row in df_top100.iterrows()
]

# Creamos los batches de BATCH_SIZE_FILTER reseñas
batches = [
    reviews_list[i : i + BATCH_SIZE_FILTER]
    for i in range(0, len(reviews_list), BATCH_SIZE_FILTER)
]

print(f"Total reseñas: {len(reviews_list)}")
print(f"Batch size:    {BATCH_SIZE_FILTER}")
print(f"Nº de batches: {len(batches)}")
print(f"Sleep entre batches: {SLEEP_FILTER}s")
print(f"Tiempo estimado: ~{len(batches) * SLEEP_FILTER / 60:.1f} min\n")

filter_results = []

for i, batch in enumerate(tqdm(batches, desc="Filtrando reseñas")):
    batch_results = call_llm_filter(batch)
    filter_results.extend(batch_results)

    # Respetamos el rate limit de Groq entre cada batch
    if i < len(batches) - 1:
        time.sleep(SLEEP_FILTER)

print(f"\n✅ Filtrado completado. Resultados obtenidos: {len(filter_results)}")

In [ ]:
# ─── Construcción del DataFrame filtrado ───────────────────────────────────────
df_filter_results = pd.DataFrame(filter_results)
df_filter_results["id"] = df_filter_results["id"].astype(int)

# Merge con el DataFrame original
df_filtered = df_top100.merge(
    df_filter_results[["id", "decision", "razon"]],
    left_on="review_id",
    right_on="id",
    how="left"
)

# Estadísticas del filtrado
counts = df_filtered["decision"].value_counts()
print("=== Resultados del filtrado ===")
print(counts.to_string())
print(f"\nTasa de relevancia: {counts.get('RELEVANTE', 0) / len(df_filtered) * 100:.1f}%")

# Nos quedamos solo con las RELEVANTES
df_relevant = df_filtered[df_filtered["decision"] == "RELEVANTE"].reset_index(drop=True)
print(f"\nReseñas que pasan al Paso 3: {len(df_relevant)}")
df_relevant[["review_id", "Valoración", "decision", "razon"]].head(10)

In [ ]:
# Visualizamos algunas reseñas que fueron descartadas (si las hay)
df_irrelevant = df_filtered[df_filtered["decision"] == "IRRELEVANTE"]
if len(df_irrelevant) > 0:
    print(f"=== Reseñas descartadas ({len(df_irrelevant)}) ===")
    for _, row in df_irrelevant.head(5).iterrows():
        print(f"\n[ID {row['review_id']}] Razón: {row['razon']}")
        print(f"Texto: {str(row['Contenido'])[:200]}...")
else:
    print("No se descartó ninguna reseña (todas son relevantes en el top-100 por longitud).")

## 4. Paso 3 — Extracción estructurada con LLM #2 (`llama-3.3-70b-versatile`)

### Diseño del prompt de extracción

**Decisiones de diseño:**
- **1 reseña por llamada**: maximiza la calidad del JSON. Con múltiples reseñas, el modelo tiende a mezclar entidades entre ellas.
- **Temperature = 0.0**: necesitamos output determinista y parseable, no creatividad.
- **Schema JSON explícito en el prompt**: reduce alucinaciones de formato.
- **Campos extraídos** (≥3 como exige el enunciado):
  1. `SentimientoGeneral`: Positivo / Negativo / Neutral
  2. `AspectosPositivos`: lista de strings
  3. `AspectosNegativos`: lista de strings
  4. `Dificultad`: categoría cerrada
  5. `GeneroPercibido`: Acción / RPG / Estrategia / etc. (campo bonus)
  6. `Recomendado`: del CSV original

**Por qué llama3-70b aquí:**  
La extracción estructurada requiere comprensión profunda del texto y generación de JSON válido y coherente. El modelo grande ofrece mucha mejor calidad en esta tarea.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """Eres un experto en análisis de reseñas de videojuegos.
Tu tarea es extraer información estructurada de una reseña y devolver ÚNICAMENTE un objeto JSON válido.
No incluyas explicaciones, markdown, ni texto adicional fuera del JSON."""


def build_extraction_prompt(review_id: int, texto: str, recomendado: int) -> str:
    """
    Construye el prompt de extracción para una sola reseña.
    Incluye el campo 'Recomendado' directamente del CSV (ground truth).
    """
    return f"""Analiza la siguiente reseña de un videojuego y extrae la información solicitada.

RESEÑA (ID: {review_id}):
\"\"\"{texto}\"\"\"

Devuelve EXACTAMENTE este JSON rellenando cada campo:
{{
  "Id": {review_id},
  "SentimientoGeneral": "<Positivo | Negativo | Neutral>",
  "AspectosPositivos": ["<aspecto1>", "<aspecto2>"],
  "AspectosNegativos": ["<aspecto1>", "<aspecto2>"],
  "Dificultad": "<Demasiado Fácil | Fácil | Equilibrado | Difícil | Demasiado Difícil | No Mencionado>",
  "GeneroPercibido": "<Acción | RPG | Estrategia | Aventura | Simulación | Deportes | Terror | Otro | No Mencionado>",
  "HorasJugadas": "<número aproximado si se menciona, o null>",
  "Recomendado": {recomendado}
}}

Reglas importantes:
- Si no se mencionan aspectos positivos o negativos, usa una lista vacía: []
- Extrae solo lo que está explícitamente en el texto, no inventes información
- Los AspectosPositivos y AspectosNegativos deben ser frases cortas y concretas"""


def call_llm_extract(review_id: int, texto: str, recomendado: int, retries: int = 3) -> dict:
    """
    Llama al LLM de extracción para una sola reseña.
    Retorna el dict con la información estructurada.
    """
    prompt = build_extraction_prompt(review_id, texto, recomendado)

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_EXTRACT,
                messages=[
                    {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                temperature=0.0,
                max_tokens=600,    # JSON de extracción no necesita más
            )
            raw_text = response.choices[0].message.content.strip()

            # Parseo robusto
            start = raw_text.find("{")
            end   = raw_text.rfind("}") + 1
            json_str = raw_text[start:end]
            result = json.loads(json_str)
            return result

        except json.JSONDecodeError as e:
            print(f"  ⚠️  JSON inválido en ID {review_id} (intento {attempt+1}/{retries}): {e}")
            time.sleep(2)
        except Exception as e:
            print(f"  ❌ Error API en ID {review_id} (intento {attempt+1}/{retries}): {e}")
            time.sleep(6)

    # Fallback: retornamos estructura vacía con el ID
    return {
        "Id": review_id,
        "SentimientoGeneral": "ERROR",
        "AspectosPositivos": [],
        "AspectosNegativos": [],
        "Dificultad": "No Mencionado",
        "GeneroPercibido": "No Mencionado",
        "HorasJugadas": None,
        "Recomendado": recomendado
    }


print("✅ Funciones de extracción definidas")
print("\n--- Ejemplo de prompt de extracción ---")
print(build_extraction_prompt(0, "El juego es brutal, llevo 200 horas y no me canso. Los jefes finales son dificilísimos pero satisfactorios.", 1))

In [ ]:
# ─── Ejecución del pipeline de extracción ──────────────────────────────────────

print(f"Reseñas a procesar: {len(df_relevant)}")
print(f"Sleep entre llamadas: {SLEEP_EXTRACT}s")
print(f"Tiempo estimado: ~{len(df_relevant) * SLEEP_EXTRACT / 60:.1f} min\n")

extraction_results = []

for i, (_, row) in enumerate(tqdm(df_relevant.iterrows(), total=len(df_relevant), desc="Extrayendo información")):
    result = call_llm_extract(
        review_id  = int(row["review_id"]),
        texto      = str(row["Contenido"]),
        recomendado= int(row["Recomendado_binario"])
    )
    extraction_results.append(result)

    # Respetamos el rate limit entre cada llamada
    if i < len(df_relevant) - 1:
        time.sleep(SLEEP_EXTRACT)

print(f"\n✅ Extracción completada. Registros obtenidos: {len(extraction_results)}")

## 5. Construcción del DataFrame final y análisis de resultados

In [ ]:
# ─── DataFrame final ───────────────────────────────────────────────────────────
df_final = pd.DataFrame(extraction_results)

print(f"Shape del resultado final: {df_final.shape}")
print(f"\nCampos extraídos: {list(df_final.columns)}")
df_final.head()

In [ ]:
# ─── Análisis de resultados ────────────────────────────────────────────────────
print("=" * 50)
print("RESUMEN DEL PIPELINE")
print("=" * 50)

print(f"\n📊 Reseñas iniciales (top-100 por longitud): {len(df_top100)}")
n_relevantes = len(df_relevant)
n_irrelevantes = len(df_top100) - n_relevantes
print(f"🔍 Tras filtrado LLM #1:")
print(f"   Relevantes:   {n_relevantes} ({n_relevantes/len(df_top100)*100:.1f}%)")
print(f"   Irrelevantes: {n_irrelevantes} ({n_irrelevantes/len(df_top100)*100:.1f}%)")
print(f"📦 Registros estructurados (LLM #2): {len(df_final)}")

print("\n--- Distribución de sentimiento general ---")
print(df_final["SentimientoGeneral"].value_counts().to_string())

print("\n--- Distribución de dificultad percibida ---")
print(df_final["Dificultad"].value_counts().to_string())

print("\n--- Géneros percibidos ---")
print(df_final["GeneroPercibido"].value_counts().to_string())

print("\n--- Correlación Sentimiento vs Recomendado ---")
print(pd.crosstab(df_final["SentimientoGeneral"], df_final["Recomendado"]))

In [ ]:
# ─── Top aspectos positivos más mencionados ────────────────────────────────────
from collections import Counter

all_positives = []
for aspects in df_final["AspectosPositivos"]:
    if isinstance(aspects, list):
        all_positives.extend([a.lower().strip() for a in aspects])

all_negatives = []
for aspects in df_final["AspectosNegativos"]:
    if isinstance(aspects, list):
        all_negatives.extend([a.lower().strip() for a in aspects])

print("=== Top 10 aspectos POSITIVOS más mencionados ===")
for aspecto, count in Counter(all_positives).most_common(10):
    print(f"  {count:3d}x  {aspecto}")

print("\n=== Top 10 aspectos NEGATIVOS más mencionados ===")
for aspecto, count in Counter(all_negatives).most_common(10):
    print(f"  {count:3d}x  {aspecto}")

## 6. Exportación del resultado final

In [ ]:
# ─── Guardar resultado estructurado ────────────────────────────────────────────
output_path = "output/analisis_videojuegos_resultados_final.csv"
df_final.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"✅ Resultado guardado en: {output_path}")
print(f"   Filas: {len(df_final)} | Columnas: {len(df_final.columns)}")

# Vista final
df_final

## 7. Reflexión final y decisiones de Prompt Engineering

### Resumen de decisiones técnicas

| Decisión | Elección | Justificación |
|---|---|---|
| Proveedor | Groq | Inferencia rápida (LPU), tier gratuito, baja latencia |
| LLM #1 (filtrado) | llama-3.1-8b-instant | Tarea binaria simple, rate limits más generosos |
| LLM #2 (extracción) | llama-3.3-70b-versatile | Mayor comprensión para extracción estructurada compleja |
| Batch size filtrado | 5 reseñas/llamada | Equilibrio entre eficiencia (menos llamadas) y control (JSON manejable) |
| Batch size extracción | 1 reseña/llamada | Máxima precisión; evita mezcla de entidades entre reseñas |
| Temperature | 0.0 en ambos | Determinismo: tareas clasificatorias, no creativas |
| Sleep filtrado | 2.5s | Por debajo del rate limit de ~30 req/min de llama3-8b |
| Sleep extracción | 4.5s | Por debajo del rate limit de ~14 req/min de llama3-70b |
| Reintentos | 3 intentos | Robustez ante errores de red o JSON malformado |

### Estrategia de prompts

**Prompt de filtrado:** Se optó por una tarea binaria (`RELEVANTE`/`IRRELEVANTE`) con criterios explícitos de qué constituye cada categoría. Se incluye el campo `razon` para auditoría y debugging del pipeline.

**Prompt de extracción:** Se usa un system prompt separado para anclar el rol del modelo, y el user prompt contiene el schema JSON exacto que se espera como output. Esto reduce drásticamente los errores de formato. Se delimita el texto de la reseña con triple comilla para separarlo claramente de las instrucciones.

### Limitaciones conocidas

- El contexto de 8192 tokens de llama3 limita el tamaño máximo de las reseñas procesables.
- Con el tier gratuito de Groq, el pipeline completo tarda ~10-15 minutos para 100 reseñas.
- La calidad de la extracción depende de la claridad del texto original de la reseña.